# Language model and sequence generation — Transformers

In this lab, we are going to study the different ways to decode text given a trained model. This step is needed for many natural language processing tasks, including translation, question answering and generation.

We will take text generation as our example: until very recently, it required more or less sophisticated fill-in-the-blank templates. The arrival of powerful models trained on large amounts of data changed the game: neural methods are now an option.

In this notebook, we will see how to generate text from a trained model, starting from the simplest method and working our way up to state-of-the-art methods. This will give you a solid understanding of the mechanisms of neural networks applied to text: the softmax activation in particular, which is at the heart of the main state-of-the-art NLP architectures.

An [excellent blog post](https://huggingface.co/blog/how-to-generate) by Hugging Face on the subject can serve as a complementary resource to this notebook.

## Installation

The [`transformers`](https://github.com/huggingface/transformers) library developed by the French-American startup [Hugging Face](https://huggingface.co/) provides a large number of models from the transformer family.

It is easily installed with `pip`:

In [ ]:
!pip install 'transformers < 5'

## Language

We are going to see how to generate text in English but also in French, by loading models pretrained on different corpora. The following cell changes the language used for the rest of the lab.

In [ ]:
lang = "en"
# lang = "fr"

## Getting a pretrained model

On top of providing the source code of many transformer architectures, the `transformers` library gives access to [many pretrained models](https://huggingface.co/transformers/pretrained_models.html).

In this lab, we are going to use the GPT2 architecture, behind the [model that surprised the NLP community](https://openai.com/blog/better-language-models/) with the quality of its generations (its successor, GPT3, has since reached [even more impressive results](https://github.com/elyase/awesome-gpt3)).

To do so, we just call the `from_pretrained` method of the class of the architecture we are interested in, with the name of the pretraining we want to get as argument. We get the tokenizer tied to the model the same way.

In [ ]:
import functools
import typing

import numpy
import tensorflow
import transformers
import tqdm.notebook


pretraining_name = "gpt2" if lang == "en" else "antoiloui/belgpt2"

model = transformers.TFGPT2LMHeadModel.from_pretrained(pretraining_name, use_safetensors=False)
tokenizer = transformers.GPT2Tokenizer.from_pretrained(pretraining_name, use_safetensors=False)

## Using the GPU

By default, TensorFlow uses the available GPUs. Let's check whether Colab gave us one:

In [ ]:
print(tensorflow.config.experimental.list_physical_devices("GPU"))

## Inspecting the model

After getting a pretrained model, it's always a good idea to check its architecture and characteristics.

In the `transformers` library, we can do so by looking at its config:

- *How many transformer layers does the model have?*
- *What are its regularization mechanisms?*
- *Its training stabilization mechanisms?*
- *We will later need the index of the special symbol used during training to delimit the start and end of texts. What is it?*

In [ ]:
print(model.config)

*Your answer here.*
-
-
-
-

### Solution

- It has 12 layers
- Its main regularization mechanism is dropout
- Its training stabilization mechanisms visible from the architecture shown above are layer normalization and residual connections.
- We can read its value in the `bos_token_id` and `eos_token_id` keys of the config: `50256`.

## The generation mechanism

To generate text from a trained model, we proceed iteratively: we generate words one by one, starting from an empty text or from a sentence we want to complete.

To generate a word, we feed the network the words generated so far (turned into embedding indices). The network then produces one score per vocabulary word. We pick one of these words before looping (appending this word to what was generated so far, then feeding the result to the network to generate the next word).

More formally, we are trying to maximize $P(\text{Text}| \text{Prompt})$ with the following decomposition:

$$P(\text{Text}| \text{Prompt}) = \Pi_{i=1}^{\text{Text length}} P(\text{Word}_i| \text{Word}_{j<i}, \text{Prompt})$$

The whole point of decoding is to find the maximal value of this product: it's the best proposal of the model. This value is impossible to compute exactly, so we resort to heuristics to approximate it.

## Tokenization

The tokenizer we got along with the model turns an input sentence into embedding indices the model can understand.

In the rest of the lab, we will use the following `encode` and `decode` functions, which turn strings into the vocabulary indices the model understands (the indices of its embedding module), and back.

In [ ]:
def encode(sentence: str) -> tensorflow.Tensor:
  tokens = tokenizer.encode(sentence,
                            add_special_tokens=False,
                            return_tensors="tf")
  bos = tensorflow.constant([[model.config.bos_token_id]], dtype="int64")
  return tensorflow.concat([bos, tensorflow.cast(tokens, "int64")], axis=1)


def decode(tokens: tensorflow.Tensor) -> str:
  if tokens.ndim == 2:
    tokens = tensorflow.squeeze(tokens)
  return tokenizer.decode(tokens[1:], clean_up_tokenization_spaces=True)

## Studying the outputs of the pretrained model

The first step to decode sentences from a trained model is to run a forward pass of the model to get its predictions for the next word. Let's study its behavior.

In Keras, to compute the forward pass of a model, we directly call `model(input_tensor)` as we have already seen.

*Study the output of the pretrained model ([documentation](https://huggingface.co/transformers/model_doc/gpt2.html#tfgpt2model)) by applying it to various sequences, for example:*

- *“”*
- *“I think therefore I am” or “Je pense donc je suis” depending on the language of the model*

*While reading the documentation, note the `past` argument: it avoids recomputing values computed during the previous decoding step. When the past argument is given to the model, the old values must not be passed again.*

In [ ]:
# Your code here

### Solution

In [ ]:
def describe_shapes(input_string: str,
                    output_tensor: tensorflow.Tensor,
                    output_past: list[tensorflow.Tensor]
                   ) -> str:
  print("-" * 80)
  print(f"Output shape for input '{input_string}': "
        f"{output_tensor.shape}")
  print(f"Size of the past: {len(output_past)}")
  print(f"Type of the past: {type(output_past[0])}")


empty_input = ""
descartes_input = ("I think, therefore I am"
                   if lang == "en"
                   else "Je pense, donc je suis")

empty_output, empty_past = model(encode(empty_input),
                                 return_dict=False)
descartes_output, descartes_past = model(encode(descartes_input),
                                         return_dict=False)

describe_shapes(empty_input, empty_output, empty_past)
describe_shapes(descartes_input, descartes_output, descartes_past)

## A `forward` function suited to decoding

Given the observations of the previous section, we will use the following `forward` function, which returns an output suited to decoding: only the last column interests us (the prediction of the next word). This `forward` function implements this selection.

In [ ]:
def forward(tokens: tensorflow.Tensor,
            past: list[tensorflow.Tensor] | None = None
           ) -> tuple[tensorflow.Tensor, list[tensorflow.Tensor]]:
  logits, new_past = model(tokens, past_key_values=past, return_dict=False)
  return logits[:, -1, :], new_past

Note that when the `past` argument is used, the tokens that were already computed must not be given again, only the new token to decode. So use either:

- `forward(all_tokens)`
- `forward(last_token, past)`

where `all_tokens` would for example be a sequence of 7 tokens of shape `(1, 7, 50257)` and `last_token` would be of shape `(1, 1, 50257)`.

*Why is it often useful to keep dimensions of size `1` in neural networks?*

*Your answer here.*

### Answer

These dimensions can often be used to compute results for several examples in parallel (batching).

## Decoding loop

A decoding loop always follows the same pattern:

1. Encode everything produced so far (inputs and previous outputs)
2. Produce the index of the most likely next word
3. Repeat 1. and 2. until a stopping criterion is met (in this lab, we will only use the number of words produced)

Step 2 is where everything happens. For now, we will code everything else, so that we can focus on step 2 afterwards.

In [ ]:
def decoding_loop(step_function) -> str:

  @functools.wraps(step_function)
  def wrapper(prompt, length, *step_args, **step_kwargs):
    token_ids = encode(prompt)
    past = None
    decoded = [token_ids]
    for i in tqdm.notebook.trange(length, desc="Words", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded += [index[None, None]]
    decoded_tensor = tensorflow.concat(decoded, axis=1)
    return decode(decoded_tensor)

  return wrapper

## Greedy decoding

Greedy decoding is the simplest method to decode text from a trained model.

At each decoding step (to decode each word), it takes the word with the highest probability.

More formally, in the equation:

$$P(\text{Text}|\text{Prompt}) = \Pi_{i=0}^{\text{Text length}} P(\text{Word}_i| \text{Word}_{j<i}, \text{Prompt})$$

Greedy decoding picks $\text{Word}_i$ to maximize $P(\text{Word}_i| \text{Word}_{j<i}, \text{Prompt})$. Yet it's often necessary to pick a less likely word to then reach very likely words.

The following example is given in the Hugging Face blog post on text generation:

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

We can see that after decoding “The”, we pick “nice” because it's the word with the highest probability for the model, and that the complete decoding “The nice woman” has a lower probability (0.20) than the decoding we would have obtained by picking “dog” (“The dog has”, probability 0.36).

*Code the `greedy(logits: tensorflow.Tensor) -> tensorflow.Tensor` function that takes the output of the model as input and selects the word to return (the index in the last dimension).*

*It will use [`tensorflow.math.argmax`](https://www.tensorflow.org/api_docs/python/tf/math/argmax). The `decoding_loop` function expects a scalar as return value: a TensorFlow tensor of shape `()`.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def greedy(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  return tensorflow.math.argmax(logits, axis=1)[0]


print(greedy("I went to the", 10))

## Testing on various inputs

*Code a `test_decoding(function, *args, **kwargs) -> None` function that takes a decoding function and its arguments as input and shows the results of this function on fixed inputs. Test the `greedy` function with it. What do you notice?*

In [ ]:
if lang == "en":
    inputs = ["I went to the",
              "I am feeling",
              "How do you",
              "How comes that",
              "I'm fine thank you, you should",
              "Superman just tweeted :",
              "President Camacho just quit !"]
else:
    inputs = ["Je suis allé au",
              "Je me sens",
              "Comment vas-tu",
              "Comment se fait-il que",
              "Ça va merci, tu devrais",
              "Superman vient de tweeter :",
              "Le président Camacho vient de démissionner !"]

In [ ]:
# Your code here

### Solution


In [ ]:
def test_decoding(function, *args, **kwargs) -> None:
  generations = []
  for i in tqdm.notebook.tqdm(inputs, desc="Prompts", leave=False):
    generations.append(function(i, *args, **kwargs))
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt: {prompt}")
    print(f"Generation: {generation}")
  print("—" * 80)

test_decoding(greedy, 50)

We notice that the answers “loop” quite quickly. It's a very common flaw of greedy decoding.

## Sampling

As we have just seen, the results of greedy decoding have many problems. They loop quickly and are often very generic. The following methods aim at fixing these shortcomings.

The first approach we are going to see consists in sampling from the results of the model instead of always picking the most likely result.

*Start from the shape of the `greedy` function but instead of picking the maximal element of the output, compute a probability distribution from the model (in log space for numerical stability), with the [`tensorflow.nn.log_softmax`](https://www.tensorflow.org/api_docs/python/tf/nn/log_softmax) activation function, and use the [`tensorflow.random.categorical`](https://www.tensorflow.org/api_docs/python/tf/random/categorical) function to sample from this distribution. The result of this sampling will be the next generated word.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def sampling(logits: tensorflow.Tensor) -> tensorflow.Tensor:
  softmaxed = tensorflow.nn.log_softmax(logits)
  return tensorflow.random.categorical(softmaxed, 1)[0][0]

test_decoding(sampling, 50)

## Top-k sampling

A variation of sampling consists in only sampling from the `k` highest results, to avoid generating a word that is too unlikely (even if it rarely happens).

*Start from the `sampling` function to code the `k_sampling(logits: tensorflow.Tensor, k: int) -> tensorflow.Tensor` function that implements this improvement. Test it with the `test_decoding` function.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def k_sampling(logits: tensorflow.Tensor, k: int) -> tensorflow.Tensor:
  topk = tensorflow.math.top_k(logits, k=k)
  softmaxed = tensorflow.nn.log_softmax(topk.values)
  sampled = tensorflow.random.categorical(softmaxed, 1)[0][0]
  return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_sampling, 50, k=20)

## Top-p sampling

Another improvement of sampling consists in only considering the first `n` outputs whose summed probability exceeds `p`, and not the following ones.

*Start from the `k_sampling` function and add top-p sampling to top-k sampling, in the `k_p_sampling(logits: tensorflow.Tensor, k: int, p: float) -> tensorflow.Tensor` function. Test it with the `test_decoding` function.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def k_p_sampling(logits: tensorflow.Tensor, k: int, p: float
                ) -> tensorflow.Tensor:
    topk = tensorflow.math.top_k(logits, k=k)
    softmaxed = tensorflow.nn.softmax(topk.values)
    current_proba_sum = 0
    j = 0
    while current_proba_sum < p:
      current_proba_sum += softmaxed[0, j]
      j += 1
    sampled = tensorflow.random.categorical(
        tensorflow.math.log(softmaxed[:, :j]), 1)[0][0]
    return tensorflow.cast(topk.indices[0, sampled], "int64")

test_decoding(k_p_sampling, length=100, k=20, p=0.85)

## Using the `transformers` functions

In the future, if you need to generate text, you can use the `generate` function of the models of the `transformers` library. Here is an example:

In [ ]:
inputs = ["Kim jong Un resign after a headache", "Donald Trump just tweeted :"]

In [ ]:
def transformers_generate(prompt: str,
                          length: int,
                          temperature: float,
                          top_k: int,
                          top_p: float) -> str:
  token_ids = encode(prompt)
  generated = model.generate(input_ids=tensorflow.cast(token_ids, "int32"),
                             max_length=len(token_ids) + length,
                             temperature=temperature,
                             top_k=top_k,
                             top_p=top_p,
                             do_sample=True,
                             num_return_sequences=1)
  return decode(tensorflow.cast(generated, "int64"))

test_decoding(transformers_generate, length=1000, temperature=1, top_k=30, top_p=0.95)